### Description of the Notebook - Governance Officer

In [35]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

### Loading the clean datasets

In [36]:
app_df = pd.read_csv('applications_clean.csv')
spend_df = pd.read_csv('spending_behavior.csv')

In [37]:
# Display the first 10 rows of the applications dataset
print("Applications Clean Dataset:")
display(app_df.head(10))

# Display the first 10 rows of the spending behavior dataset
print("\nSpending Behavior Dataset:")
display(spend_df.head(10))

Applications Clean Dataset:


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,...,financials_savings_balance,decision_loan_approved,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,...,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,...,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,...,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,...,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,...,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019.0,110000.0,...,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022.0,55000.0,...,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,NaN,NaN
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223.0,82000.0,...,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,NaN,NaN
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044.0,69000.0,...,15974,False,algorithm_risk_score,NaN,NaN,NaN,RESUBMISSION,NaN,NaN,NaN
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080.0,55000.0,...,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Spending Behavior Dataset:


,app_id,category,amount
0,app_200,Shopping,480
1,app_200,Rent,790
2,app_200,Alcohol,247
3,app_037,Rent,608
4,app_037,Dining,96
5,app_037,Healthcare,243
6,app_215,Rent,109
7,app_024,Fitness,575
8,app_184,Entertainment,463
9,app_275,Entertainment,571


In [38]:
merged_df = pd.merge(app_df, spend_df, on='app_id', how='left')

In [46]:
# Display the first 10 rows of the applications dataset
print("Merged Dataset:")
display(merged_df.head(10))

Merged Dataset:


,app_id,processing_timestamp,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,...,decision_rejection_reason,loan_purpose,decision_interest_rate,decision_approved_amount,notes,data_quality_flag,missing_fields,flag_note,category,amount
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shopping,480
1,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rent,790
2,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alcohol,247
3,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rent,608
4,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dining,96
5,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Healthcare,243
6,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,...,NaN,vacation,3.7,59000.0,NaN,NaN,NaN,NaN,Rent,109
7,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,...,NaN,NaN,4.3,34000.0,NaN,NaN,NaN,NaN,Fitness,575
8,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Entertainment,463
9,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019.0,110000.0,...,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Entertainment,571


In [40]:
# Create a summary dataframe with unique categories and their absolute frequencies
category_summary = merged_df['category'].value_counts().reset_index()

# Rename columns for clarity
category_summary.columns = ['Spending Behaviour', 'Absolute Frequency']

# Display the resulting dataframe
print("Summary of Spending Categories:")
display(category_summary)

Summary of Spending Categories:


,Spending Behaviour,Absolute Frequency
0,Travel,80
1,Utilities,76
2,Entertainment,72
3,Fitness,70
4,Healthcare,68
5,Insurance,67
6,Dining,65
7,Groceries,65
8,Education,64
9,Transportation,61


# PII IDENTIFICATION 

The given dataset contains the following PIIs:

**Direct Identifiers:**
- **Name, Email, and Social Security Number (SSN)** are direct identifiers as they can be uniquely and obviously linked to an individual without combining any further data.

**Quasi-identifiers:** This category encompasses all attributes that can lead to the identification of data subjects when combined with other fields; these identifiers, by themselves, do not lead to the obvious and direct identification of a person.
- **Gender, Date of Birth, and ZIP Code** are typical quasi-identifiers.
- **Annual Income, Savings Balance, Approved Loan Amount, and Interest Rate** are financial attributes that can be combined to identify a specific individual.
- **IP Address:** IP addresses fall into the quasi-identifier category because, on their own, they cannot lead to the direct identification of a data subject.

**Spending behavior** is not identified as a Personally Identifiable Information (PII) identifier, as its level of granularity appears too broad to lead to the identification of a data subject. It must be noted, however, that if NovaCred possessed more granular and capillary spending data, this category would certainly be included in the list of PIIs. Additionally, from the list of unique spending behaviors, we can identify  sensitive categories (Alcohol, Gambling, and Adult Entertainment) inside of this variable that clearly violate Article 9 of the GDPR, which protects the processing of sensitive data. This violation will be addressed later in a specific section of the GDPR mapping.

# GDPR MAPPING

In the following sections, Novacred data governance will be evaluated under the GDPR principles stated in Article 5.

### Lawfullness

**Lawfulness:** processing of personal data carried out by a controller must have a legal basis under the GDPR. In this particular business case, the **legal basis is the execution of a contract:** Novacred can legitimately store personal data for the potential execution of a loan contract requested by the client itself. Therefore, under this partuclar circumstance no violations can be found.

### Fairness

**Fairness**: the fairness principle states that the processing of personal data must be fair towards the individual whose personal data are concerned. As we discovered in the bias analysis part several biases have been found in the model, therefore we can confidently say that this principle was not respected.

### Transparency

**Transparency**: the transparency principle states that data controllers must provide individuals with clear information regarding the processing of their personal data before they are given. We cannot assess whether thsi principle is respected or not, as we do not dispose of any information in this regard. 

### Purpose limitation
The purpose limitation principle states that personal data must be collected for specified, explicit, and legitimate purposes. Among the PII attributes outlined above, the IP address stands out as a variable that is not strictly necessary for the credit risk assessment and loan concession process: consequently, its inclusion violates the purpose limitation principle.

To ensure compliance, IP addresses have been eliminated from the dataset, and future data ingestion processes should be modified to no longer request IP addresses from clients.

In [41]:
# --- GOVERNANCE OFFICER TASK: DATA MINIMIZATION ---
# Removing the IP address as it violates GDPR Art. 5(1)(c)

if 'applicant_info_ip_address' in app_df.columns:
    app_df = app_df.drop(columns=['applicant_info_ip_address'])
    print("SUCCESS: 'applicant_info_ip_address' has been removed.")
else:
    print("NOTE: Column not found or already removed.")

# Verify the current columns
print("\nCurrent columns in app_df:")
print(app_df.columns.tolist())

SUCCESS: 'applicant_info_ip_address' has been removed.

Current columns in app_df:
['app_id', 'processing_timestamp', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'loan_purpose', 'decision_interest_rate', 'decision_approved_amount', 'notes', 'data_quality_flag', 'missing_fields', 'flag_note']


### Data minimisation

### Accuracy

### Storage Limitation

The storage limitation principle dictates that personal data should be kept in a form which permits identification of data subjects for no longer than is necessary for the purposes for which the personal data are processed. Furthermore, explicit time limits must be established for the automatic erasure of customer records.

Currently, NovaCred lacks any defined time limits; all data is stored indefinitely, which represents a clear violation of this principle. To ensure compliance, the company must define a reasonable retention threshold (e.g., 1 year), after which data is subject to automatic deletion. In this regard, a distinction must be made based on the outcome of the application:

Approved Loans: The retention countdown shall only commence once the customer's legal and financial obligations toward the company are fully extinguished and recorded in the system. For instance, data should be deleted one year after the final loan installment has been successfully processed.

Denied Loans: For applicants whose loans were rejected, the one-year countdown shall start from the date the rejection notification is issued. In the event of a new submission, the countdown is reset until the issuance of a new decision.

In [42]:
import pandas as pd
import numpy as np

# --- GOVERNANCE OFFICER TASK: DATA RETENTION & STORAGE LIMITATION ---
# Implementing automatic deletion triggers based on loan outcome (Art. 5.1.e GDPR)

# Assuming 'application_date' exists; if not, we use a placeholder for demonstration
if 'application_date' not in app_df.columns:
    app_df['application_date'] = pd.to_datetime('2024-01-01')
else:
    app_df['application_date'] = pd.to_datetime(app_df['application_date'])

# 1. Create 'loan_rejection_date'
# This field is populated only if the loan was denied
app_df['loan_rejection_date'] = pd.NaT
app_df.loc[app_df['decision_loan_approved'] == False, 'loan_rejection_date'] = app_df['application_date']

# 2. Create 'final_installment_date'
# For approved loans, we simulate the end of the contract (e.g., 2 years after application)
app_df['final_installment_date'] = pd.NaT
app_df.loc[app_df['decision_loan_approved'] == True, 'final_installment_date'] = \
    app_df['application_date'] + pd.DateOffset(years=2)

# 3. Create 'automatic_deletion_date'
# Adding a 365-day retention threshold (1 year) to the available trigger date
app_df['automatic_deletion_date'] = \
    app_df[['loan_rejection_date', 'final_installment_date']].max(axis=1) + pd.Timedelta(days=365)

# Verification: display the new retention management columns
print("Data Retention Audit Trail:")
display(app_df[['decision_loan_approved', 'loan_rejection_date', 'final_installment_date', 'automatic_deletion_date']].head(10))

Data Retention Audit Trail:


,decision_loan_approved,loan_rejection_date,final_installment_date,automatic_deletion_date
0,False,2024-01-01,NaT,2024-12-31
1,False,2024-01-01,NaT,2024-12-31
2,True,NaT,2026-01-01,2027-01-01
3,True,NaT,2026-01-01,2027-01-01
4,False,2024-01-01,NaT,2024-12-31
5,False,2024-01-01,NaT,2024-12-31
6,True,NaT,2026-01-01,2027-01-01
7,True,NaT,2026-01-01,2027-01-01
8,False,2024-01-01,NaT,2024-12-31
9,False,2024-01-01,NaT,2024-12-31


### Integrity and confidentiality

**Violation of the principle of integrity:** the personal data managed by NovaCred are accessible in their original form, without any type of appropriate technical measure being implemented to secure the data. This deficiency represents a clear violation of Articles 25 and 32 of the GDPR, which oblige controllers to implement a range of protective measures to assure data safety.

### Article 9: processing of sensitive data

As anticipated in the PII Identification section, the variable Spending Behavior attribute contains several categories that must be considered with particular attention, as they fall, directly or indirectly, into the definition of senstive information according to article 9 of GDPR:

- **Healthcare:** This category is directly identifiable as carrying sensitive information, as it reveals an individual's health status. Spending patterns in this category can be clearly linked to a data subject having specific health conditions.
- **Alcohol:** This spending behavior can be indirectly linked to a person's health status; it is widely recognized that alcohol consumption is associated with specific health risks and conditions.
- **Adult Entertainment:** This category constitutes sensitive data as it reveals information regarding a person's sex life or sexual orientation.
- **Gambling:** Gambling is associated with addictions that have repercussions on a person's physical and mental state. Therefore, it can be affirmed that gambling behavior may reveal clear information about an individual's health status, which falls under the categories of data explicitly defined as sensitive by the GDPR.

NovaCred's data processing does not fall into the legal exceptions under which sensitive information can be collected without explicit consent; therefore, it must be assessed whether explicit consent was provided by clients at the moment of submission. The provided data completely **lacks a consent tracking mechanism**, which represents a **clear violation of Article 9 of the GDPR.** To restore the lawfulness of the processed data, the following operations will be performed:

- **Deletion of historical sensitive data:** NovaCred is not subject to an absolute prohibition on using this sensitive data in loan granting decisions; this data can be used, provided that explicit consent is obtained. Therefore, the sensitive categories themselves will not be eliminated from the system; only the specific sensitive records collected in the past in violation of the law will be deleted.
- **Establishment of an explicit consent procedure:** Creation of a formal explicit consent request procedure: during the loan application submission phase, the client must be explicitly asked whether they grant their consent for the processing of sensitive data.
- **Implementation of a consent tracking attribute:** Creation of a new 'tracking_customer_consent' column: True if consent was given, False if not.

In [43]:
# 1. Defining sensitive spending behaviour categories
sensitive_categories = ['Healthcare', 'Alcohol', 'Adult Entertainment', 'Gambling']

# 2. Historical data deletion for the identified categories
spend_df_cleaned = spend_df[~spend_df['category'].isin(sensitive_categories)].copy()

print(f"Sensible records removed: {len(spend_df) - len(spend_df_cleaned)}")

Sensible records removed: 91


In [44]:
# Data check: absolute frequency in each spending behaviour category
category_summary = spend_df_cleaned['category'].value_counts().reset_index()

# Rename columns for clarity
category_summary.columns = ['Spending Behaviour', 'Absolute Frequency']

# Display the resulting dataframe
print("Summary of Spending Categories:")
display(category_summary)

Summary of Spending Categories:


,Spending Behaviour,Absolute Frequency
0,Travel,80
1,Utilities,76
2,Entertainment,72
3,Fitness,70
4,Insurance,67
5,Dining,65
6,Groceries,65
7,Education,64
8,Transportation,61
9,Rent,59


In [45]:
# Creating the customer explicit consent column
# Every record is set to false as there is no track of any explicit consent given for the historical data in the dataset
app_df['tracking_customer_consent'] = False

# Check
display(app_df[['app_id', 'tracking_customer_consent']].head(10))

,app_id,tracking_customer_consent
0,app_200,False
1,app_037,False
2,app_215,False
3,app_024,False
4,app_184,False
5,app_275,False
6,app_099,False
7,app_246,False
8,app_042,False
9,app_348,False


# AI ACT MAPPING

### Risk Classification

### Obligations